# Grummon et al. (2023) front-of-package labeling RCT preprocessing

This notebook implements the data preprocessing pipeline for Grummon et al. (2023, JAMA Network Open) from the official ResearchBox 1895 replication archive while preserving strict isolation from the Grummon et al. (2025) evaluation benchmark.

## Step 1. Raw ingestion, header normalization, and cohort filtering

We load the raw Qualtrics CSV export, normalize the six raw header discrepancies relative to the official Stata cleaning script, replace `-99` missing sentinels with `NaN`, apply the sequential CONSORT eligibility rules (`global toofast 191.2` from soft-launch median `573.5 / 3` seconds), repair the documented `snack_healthyY_W == 10` Qualtrics coding typo, unify all 32 arm-specific outcome columns with zero cross-arm leakage checks, and join the official 16-beverage and 16-snack nutrition lookup tables.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

GRUMMON2023_DIR = (
    Path(".").resolve()
    if Path("ResearchBox 1895").is_dir()
    else Path("code/data/nutrition/grummon2023").resolve()
)
RB_DATA_DIR = GRUMMON2023_DIR / "ResearchBox 1895" / "Data"
RAW_CSV = RB_DATA_DIR / "VLO-Survey-2022-3+FINAL_December+5,+2022_09.36-NoIDs.csv"
BEV_DTA = RB_DATA_DIR / "BeverageCharacteristics.dta"
SNACK_DTA = RB_DATA_DIR / "SnackCharacteristics.dta"

raw_df = pd.read_csv(RAW_CSV, low_memory=False)
if len(raw_df) != 10_055:
    raise ValueError(f"Expected 10,055 raw Qualtrics rows, got {len(raw_df)}")

HEADER_FIXES = {
    "snack _healthyY_Cal": "snack_healthyY_Cal",
    "snack _healthyG_Cal": "snack_healthyG_Cal",
    "stigma_personal_CAL": "stigma_personal_Cal",
    "stigma_obesity_CAL": "stigma_obesity_Cal",
    "stigma_blame_CAL": "stigma_blame_Cal",
    "disgust_CAL": "disgust_Cal",
}
missing_raw_headers = [col for col in HEADER_FIXES if col not in raw_df.columns]
if missing_raw_headers:
    raise KeyError(f"Missing expected raw columns: {missing_raw_headers}")

df_all = raw_df.rename(columns=HEADER_FIXES).replace([-99, "-99", -99.0], np.nan)
for num_col in ["Q_TotalDuration", "confirmkeepdata", "consent", "age", "foplcond"]:
    df_all[num_col] = pd.to_numeric(df_all[num_col], errors="raise")

# Sequential CONSORT exclusions matching 00_AnalysisParentFile.do (toofast = 573.5 / 3 = 191.2s) and 01_DataPrep.do
TOOFAST_SEC = 191.2
step_complete = df_all[df_all["svy_status"] == "complete"]
step_not_speeder = step_complete[step_complete["Q_TotalDuration"] >= TOOFAST_SEC]
step_keep = step_not_speeder[step_not_speeder["confirmkeepdata"] != 0]
step_consent = step_keep[step_keep["consent"] != 0]
clean_df = step_consent[step_consent["age"] >= 18].copy().reset_index(drop=True)

if len(clean_df) != 7_945:
    raise ValueError(f"Expected N=7,945 analytic respondents, got {len(clean_df)}")

clean_df["foplcond"] = clean_df["foplcond"].astype(int)
ARM_SUFFIX = {1: "Cal", 2: "G", 3: "TLL", 4: "PA", 5: "W"}
ARM_NAMES = {
    1: "Calorie",
    2: "Green",
    3: "TrafficLight",
    4: "PhysicalActivity",
    5: "Warning",
}
if set(clean_df["foplcond"].unique()) != set(ARM_SUFFIX):
    raise ValueError(f"Unexpected foplcond values: {sorted(clean_df['foplcond'].unique())}")

clean_df["arm_suffix"] = clean_df["foplcond"].map(ARM_SUFFIX)
clean_df["arm"] = clean_df["foplcond"].map(ARM_NAMES)
clean_df["CASEID"] = [f"grummon2023_{i:04d}" for i in range(1, len(clean_df) + 1)]

# Fix documented Qualtrics recode typo in 01_DataPrep.do (line 117): 10 -> 5 ("Very healthy")
clean_df["snack_healthyY_W"] = pd.to_numeric(clean_df["snack_healthyY_W"], errors="raise")
n_snack_typo = int((clean_df["snack_healthyY_W"] == 10).sum())
if n_snack_typo != 55:
    raise ValueError(f"Expected 55 rows with snack_healthyY_W == 10, got {n_snack_typo}")
clean_df.loc[clean_df["snack_healthyY_W"] == 10, "snack_healthyY_W"] = 5

OUTCOME_STEMS = [
    "bev_choice", "snack_choice",
    "noticed_cal", "noticed_trial",
    "infl_bev", "help_bev", "infl_snack", "help_snack",
    "attn", "elab", "pme_disc", "pme_enc",
    "affect_worried", "affect_scared", "affect_guilty", "affect_ashamed", "affect_sad",
    "bev_healthyR", "bev_healthyY", "bev_healthyG",
    "snack_healthyR", "snack_healthyY", "snack_healthyG",
    "learn", "trust", "control", "react_manip", "react_annoy",
    "stigma_personal", "stigma_obesity", "stigma_blame", "disgust",
]

for stem in OUTCOME_STEMS:
    merged_col = pd.Series(np.nan, index=clean_df.index, dtype="float64")
    for cond_code, sfx in ARM_SUFFIX.items():
        arm_col = f"{stem}_{sfx}"
        if arm_col not in clean_df.columns:
            raise KeyError(f"Missing expected arm column {arm_col!r}")
        in_arm = clean_df["foplcond"] == cond_code
        leaked = clean_df.loc[~in_arm, arm_col].dropna()
        if len(leaked) != 0:
            raise ValueError(f"Cross-arm data leakage in {arm_col}: {len(leaked)} non-null values outside arm {cond_code}")
        merged_col.loc[in_arm] = pd.to_numeric(clean_df.loc[in_arm, arm_col], errors="raise")
    clean_df[stem] = merged_col

bev_chars = pd.read_stata(BEV_DTA, convert_categoricals=False).sort_values("itemnumber_bev").reset_index(drop=True)
snack_chars = pd.read_stata(SNACK_DTA, convert_categoricals=False).sort_values("itemnumber_snack").reset_index(drop=True)
if len(bev_chars) != 16 or len(snack_chars) != 16:
    raise ValueError(f"Expected 16 beverages and 16 snacks, got {len(bev_chars)} and {len(snack_chars)}")

clean_df = clean_df.merge(bev_chars, left_on="bev_choice", right_on="itemnumber_bev", how="left")
clean_df = clean_df.merge(
    snack_chars,
    left_on="snack_choice",
    right_on="itemnumber_snack",
    how="left",
    suffixes=("_bev_meta", "_snack_meta"),
)

total_nonmissing = int(clean_df[OUTCOME_STEMS].notna().sum().sum())
print(
    f"Raw rows: {len(raw_df):,} -> complete: {len(step_complete):,} -> "
    f"non-speeder (>={TOOFAST_SEC}s): {len(step_not_speeder):,} -> "
    f"analytic cohort: {len(clean_df):,}"
)
print("Arm counts:", clean_df.groupby(["foplcond", "arm"]).size().to_dict())
print(f"Unified {len(OUTCOME_STEMS)} outcome stems ({total_nonmissing:,} non-missing responses; repaired {n_snack_typo} snack_healthyY_W=10 rows)")


Raw rows: 10,055 -> complete: 8,145 -> non-speeder (>=191.2s): 8,110 -> analytic cohort: 7,945
Arm counts: {(1, 'Calorie'): 1593, (2, 'Green'): 1587, (3, 'TrafficLight'): 1588, (4, 'PhysicalActivity'): 1590, (5, 'Warning'): 1587}
Unified 32 outcome stems (254,145 non-missing responses; repaired 55 snack_healthyY_W=10 rows)


## Step 2. Verification against published tables and test-set isolation

We verify the cleaned cohort against the published numbers in Figure 2, Table 1, Table 2, eTable 5, and eTable 6 of Grummon et al. (2023), and confirm zero respondent, item, or product overlap with the Grummon et al. (2025) evaluation benchmark (`grummon_US_long.parquet`).

In [2]:
GRUMMON2025_PARQUET = (
    GRUMMON2023_DIR.parent
    / "Nutrition Info and Other Front-of-Package Labels and Simulated Food and Beverage Purchases"
    / "grummon_US_long.parquet"
)

# 1. CONSORT Figure 2 arm sample sizes
expected_arm_n = {1: 1593, 2: 1587, 3: 1588, 4: 1590, 5: 1587}
actual_arm_n = clean_df["foplcond"].value_counts().sort_index().to_dict()
if actual_arm_n != expected_arm_n:
    raise AssertionError(f"CONSORT arm count mismatch: {actual_arm_n} vs {expected_arm_n}")

# 2. Table 1 participant characteristics (exact published counts across all 5 arms)
gender_num = pd.to_numeric(clean_df["gender"], errors="raise")
latino_num = pd.to_numeric(clean_df["latino"], errors="raise")
educ_num = pd.to_numeric(clean_df["educ"], errors="raise")
nfp_num = pd.to_numeric(clean_df["NFPuse"], errors="raise")
diet_num = pd.to_numeric(clean_df["dietqual"], errors="raise")
hh_num = pd.to_numeric(clean_df["hhsize_num"], errors="raise")

table1_checks = {
    "Age 18-29": (clean_df["age"].between(18, 29), [310, 311, 324, 317, 343]),
    "Age 30-44": (clean_df["age"].between(30, 44), [419, 382, 402, 408, 408]),
    "Age 45-59": (clean_df["age"].between(45, 59), [426, 394, 385, 374, 389]),
    "Age >=60": (clean_df["age"] >= 60, [438, 500, 477, 491, 447]),
    "Gender Female (1)": (gender_num == 1, [814, 799, 825, 844, 796]),
    "Gender Male (2)": (gender_num == 2, [757, 773, 746, 729, 774]),
    "Gender Nonbinary/Other (3,4)": (gender_num.isin([3, 4]), [22, 15, 17, 17, 17]),
    "Latino/Hispanic": (latino_num == 1, [295, 235, 270, 246, 251]),
    "Educ HS or less": (educ_num.isin([1, 2]), [417, 401, 426, 437, 436]),
    "Educ Some college": (educ_num == 3, [409, 361, 337, 367, 340]),
    "Educ Associate/Bachelor": (educ_num.isin([4, 5]), [596, 650, 639, 627, 614]),
    "Educ Graduate": (educ_num == 6, [171, 175, 186, 159, 197]),
    "NFP Never": (nfp_num == 1, [194, 166, 195, 169, 180]),
    "NFP Always": (nfp_num == 5, [160, 157, 138, 163, 164]),
    "Diet Poor": (diet_num == 1, [97, 75, 99, 101, 73]),
    "HH size 1": (hh_num == 1, [342, 386, 351, 340, 342]),
    "HH size >=4": (hh_num.between(4, 20), [405, 361, 387, 378, 404]),
}
for label, (mask, expected_counts) in table1_checks.items():
    got = [int((mask & (clean_df["foplcond"] == c)).sum()) for c in [1, 2, 3, 4, 5]]
    if got != expected_counts:
        raise AssertionError(f"Table 1 mismatch for {label}: got {got}, expected {expected_counts}")

# 3. eTable 5 & eTable 6 primary/secondary selection outcomes (calories, sugar, sodium, satfat, G/Y/R shares, and ADEs)
expected_etable5 = {
    "item_cal_bev": [(125.8, 116.8), (91.7, 113.6), (94.3, 115.3), (86.8, 111.7), (97.6, 113.9)],
    "item_sug_bev": [(32.9, 31.2), (23.8, 30.0), (24.4, 30.4), (22.4, 29.5), (25.4, 30.3)],
    "item_sodium_bev": [(93.8, 82.4), (81.0, 85.1), (70.5, 81.6), (87.9, 88.3), (93.0, 87.6)],
    "item_cal_snack": [(182.8, 69.2), (170.1, 61.6), (169.1, 62.9), (164.3, 68.4), (168.6, 67.2)],
    "item_sug_snack": [(11.5, 14.4), (9.2, 13.2), (8.4, 12.7), (9.2, 13.1), (8.7, 13.5)],
    "item_sodium_snack": [(149.6, 74.1), (152.5, 66.8), (153.2, 68.0), (142.0, 72.8), (144.0, 73.9)],
    "item_satfat_snack": [(2.2, 1.9), (1.8, 1.8), (1.6, 1.7), (1.7, 1.8), (1.6, 1.8)],
}
for col, exp_list in expected_etable5.items():
    got_list = [
        (
            round(float(clean_df.loc[clean_df["foplcond"] == c, col].mean()), 1),
            round(float(clean_df.loc[clean_df["foplcond"] == c, col].std(ddof=1)), 1),
        )
        for c in [1, 2, 3, 4, 5]
    ]
    if got_list != exp_list:
        raise AssertionError(f"eTable 5 mismatch for {col}: got {got_list}, expected {exp_list}")

expected_ades = {
    "item_cal_bev": [-34.2, -31.5, -39.0, -28.2],
    "item_cal_snack": [-12.7, -13.7, -18.5, -14.2],
}
for col, exp_ade in expected_ades.items():
    ctrl_mean = float(clean_df.loc[clean_df["foplcond"] == 1, col].mean())
    got_ade = [
        round(float(clean_df.loc[clean_df["foplcond"] == c, col].mean()) - ctrl_mean, 1)
        for c in [2, 3, 4, 5]
    ]
    if got_ade != exp_ade:
        raise AssertionError(f"eTable 6 ADE mismatch for {col}: got {got_ade}, expected {exp_ade}")

expected_tier_counts = {
    ("item_col_str_jo_bev_meta", "G"): [311, 526, 601, 473, 412],
    ("item_col_str_jo_bev_meta", "Y"): [459, 471, 420, 564, 549],
    ("item_col_str_jo_bev_meta", "R"): [822, 590, 567, 552, 626],
    ("item_col_str_jo_snack_meta", "G"): [268, 480, 494, 384, 427],
    ("item_col_str_jo_snack_meta", "Y"): [677, 619, 673, 733, 706],
    ("item_col_str_jo_snack_meta", "R"): [648, 488, 421, 473, 454],
}
for (col, tier), exp_cnts in expected_tier_counts.items():
    got_cnts = [int(((clean_df["foplcond"] == c) & (clean_df[col] == tier)).sum()) for c in [1, 2, 3, 4, 5]]
    if got_cnts != exp_cnts:
        raise AssertionError(f"eTable 5 tier count mismatch for {(col, tier)}: got {got_cnts}, expected {exp_cnts}")

# 4. Table 2 healthfulness perceptions, stigma, and positive label perceptions
expected_table2_means = {
    "bev_healthyG": [3.8, 4.0, 4.0, 4.0, 4.0],
    "bev_healthyY": [3.7, 3.5, 3.3, 3.6, 3.7],
    "bev_healthyR": [3.4, 3.2, 2.7, 3.0, 2.8],
    "snack_healthyG": [3.3, 3.4, 3.6, 2.9, 3.1],
    "snack_healthyY": [2.7, 2.7, 2.7, 2.6, 2.4],
    "snack_healthyR": [2.3, 2.3, 2.0, 2.2, 2.0],
    "stigma_personal": [1.9, 1.8, 2.1, 2.0, 2.1],
    "disgust": [2.0, 1.9, 2.0, 1.9, 2.0],
    "learn": [2.7, 2.8, 3.1, 3.5, 3.2],
    "trust": [3.8, 3.5, 3.6, 3.7, 3.8],
}
for col, exp_means in expected_table2_means.items():
    got_means = [round(float(clean_df.loc[clean_df["foplcond"] == c, col].mean()), 1) for c in [1, 2, 3, 4, 5]]
    if got_means != exp_means:
        raise AssertionError(f"Table 2 mean mismatch for {col}: got {got_means}, expected {exp_means}")

# 5. Strict test-set isolation against Grummon 2025 (grummon_US_long.parquet)
g2025_df = pd.read_parquet(GRUMMON2025_PARQUET, columns=["CASEID", "arm", "v", "question", "option_texts"])
if not set(clean_df["CASEID"]).isdisjoint(set(g2025_df["CASEID"])):
    raise AssertionError("CASEID collision with Grummon 2025 test set")
if not set(OUTCOME_STEMS).isdisjoint(set(g2025_df["v"].unique())):
    raise AssertionError("Outcome stem collision with Grummon 2025 v codes")
if not set(clean_df["arm"].unique()).isdisjoint(set(g2025_df["arm"].unique())):
    raise AssertionError("Arm name collision with Grummon 2025 arms")

g2023_products = set(bev_chars["label_bev"].str.strip()) | set(snack_chars["label_snack"].str.strip())
g2025_products = {opt for opts in g2025_df["option_texts"] for opt in opts if opt != "They're the same"}
if not g2023_products.isdisjoint(g2025_products):
    raise AssertionError(f"Product overlap with Grummon 2025: {g2023_products & g2025_products}")

print("Verified Figure 2 CONSORT, Table 1 (17 rows), eTable 5 (7 continuous + 6 tier rows), eTable 6 ADEs, and Table 2 (10 rows): 100% exact match.")
print(
    f"Verified zero overlap with grummon_US_long.parquet "
    f"({g2025_df['CASEID'].nunique():,} 2025 respondents, {len(g2025_products)} 2025 products vs {len(g2023_products)} 2023 products)."
)


Verified Figure 2 CONSORT, Table 1 (17 rows), eTable 5 (7 continuous + 6 tier rows), eTable 6 ADEs, and Table 2 (10 rows): 100% exact match.
Verified zero overlap with grummon_US_long.parquet (5,633 2025 respondents, 12 2025 products vs 32 2023 products).


## Step 3. Survey item catalog, Grummon 2025 overlap tags, and audit interface artifacts

We export all 54 survey items and display blocks from `VLOSurveyItems.docx` into `question_audit.csv`, `.audit_details.json`, and `question_tags.json` so the entire instrument is inspectable in the shared Next.js audit workspace alongside empirical arm-level response distributions. Items that also appear in the unexported secondary survey of Grummon et al. (2025) receive explicit `in-grummon2025` or `grummon2025-similar` tags, shared demographic covariates receive `in-grummon2025-persona`, and retrospective recall items receive `prior-task-dep`.

In [3]:
import importlib
import sys

if str(GRUMMON2023_DIR) not in sys.path:
    sys.path.insert(0, str(GRUMMON2023_DIR))

import stimulus
importlib.reload(stimulus)

csv_path, tags_path, details_path = stimulus.write_grummon2023_audit_files(GRUMMON2023_DIR)
audit_df, tags_by_key, details_payload = stimulus.build_survey_audit_artifacts(clean_df, bev_chars, snack_chars)

if len(audit_df) != 54:
    raise ValueError(f"Expected 54 survey catalog rows, got {len(audit_df)}")
if int((audit_df["include"] == "True").sum()) != 26:
    raise ValueError(f"Expected 26 default-included outcome items, got {(audit_df['include'] == 'True').sum()}")

for stem in stimulus.OUTCOME_STEMS:
    matched = audit_df[audit_df["QuestionID"] == stem]
    if len(matched) != 1:
        raise ValueError(f"Expected 1 audit row for {stem}, got {len(matched)}")
    row_key = f"{matched.iloc[0]['catalog_idx']}:{stem}"
    pvs = details_payload["items"][row_key]["paired_variants"]
    if len(pvs) != 5:
        raise ValueError(f"{stem}: expected 5 arm variants, got {len(pvs)}")
    sum_variant_n = sum(int(pv["n"]) for pv in pvs)
    expected_n = int(clean_df[stem].notna().sum())
    if sum_variant_n != expected_n:
        raise ValueError(f"{stem}: variant N sum {sum_variant_n} != clean_df non-null {expected_n}")

print(f"Wrote {csv_path.name} ({len(audit_df)} rows; {(audit_df['include'] == 'True').sum()} included; {len(tags_by_key)} tagged)")
for k, v in tags_by_key.items():
    print(f"  {k:24s} -> {v}")

Wrote question_audit.csv (54 rows; 26 included; 23 tagged)
  11:noticed_cal           -> ['prior-task-dep']
  12:noticed_trial         -> ['prior-task-dep', 'in-grummon2025']
  13:infl_bev              -> ['prior-task-dep', 'grummon2025-similar']
  14:help_bev              -> ['prior-task-dep', 'grummon2025-similar']
  15:infl_snack            -> ['prior-task-dep', 'grummon2025-similar']
  16:help_snack            -> ['prior-task-dep', 'grummon2025-similar']
  18:elab                  -> ['grummon2025-similar']
  21:affect_worried        -> ['in-grummon2025']
  22:affect_scared         -> ['in-grummon2025']
  23:affect_guilty         -> ['in-grummon2025']
  24:affect_ashamed        -> ['in-grummon2025']
  25:affect_sad            -> ['in-grummon2025']
  27:trust                 -> ['in-grummon2025']
  28:control               -> ['kev-type-check']
  35:age                   -> ['in-grummon2025-persona']
  36:gender                -> ['in-grummon2025-persona']
  37:latino               

## Step 4. Arm-specific visual stimulus reconstruction and Kev encoding verification

We reconstruct the text counterpart of what human participants viewed in each of the five trial arms across the vending selection tasks, the six single-product healthfulness ratings, and the label reaction items, including the arm-specific digital posters from `eFigure`, the `0 CALORIES PER SERVING` image anomaly documented for `Diet Mountain Dew - 20oz` in `eTable 1` Footnote a, and singular versus plural label stems. Every `(arm, item)` prompt is verified directly through `KevEncoder.prepare` (`strict=True`, zero truncation).

In [4]:
CODE_DIR = GRUMMON2023_DIR.parents[2]
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from models.kev_pointer import KevEncoder

CKPT_FOR_TOKENIZER = CODE_DIR / "runs" / "kev-0.8b-resp-issp+twin2k-grouped-newvalid" / "best"
kev_enc = KevEncoder.for_base(str(CKPT_FOR_TOKENIZER))
product_catalog = stimulus.build_product_catalog(bev_chars, snack_chars)

sample_state = "\n".join([
    "Country: United States",
    "Sex: Female",
    "Age: 30-49",
    "Education: Associate's degree",
    "Income: $50,000 to $74,999",
    "Race: White",
    "Hispanic or Latino: No",
])

token_rows = []
for stem in stimulus.OUTCOME_STEMS:
    audit_row = audit_df[audit_df["QuestionID"] == stem].iloc[0]
    q_type = audit_row["kev_type"]
    for _, _, arm_name in stimulus.ARM_ORDER:
        spec = stimulus.get_arm_variant_spec(stem, arm_name, product_catalog)
        instructions = f"{spec['context']}\n\n{spec['question_stem']}" if spec["context"] else spec["question_stem"]
        opts = spec["option_texts"]
        descs = spec["option_descriptions"]
        if q_type == "choice":
            criteria = dict(zip(opts, descs, strict=True)) if descs is not None else {o: None for o in opts}
            target = {opts[0]: 1.0, **{o: 0.0 for o in opts[1:]}}
        elif q_type == "score":
            criteria = opts
            target = {"0": 1.0, **{str(i): 0.0 for i in range(1, len(opts))}}
        else:
            raise ValueError(f"Unexpected kev_type {q_type!r} on {stem}")

        view = {
            "state": sample_state,
            "questions": {
                f"{arm_name}:{stem}": {
                    "type": q_type,
                    "instructions": instructions,
                    "criteria": criteria,
                    "target": target,
                }
            },
        }
        encoded, _ = kev_enc.prepare(view)
        token_rows.append({
            "stem": stem,
            "arm": arm_name,
            "included": audit_row["include"] == "True",
            "state_tokens": int(encoded["state_tokens"]),
            "branch_tokens": int(len(encoded["ids"]) - encoded["state_tokens"]),
            "total_tokens": int(len(encoded["ids"])),
        })

token_df = pd.DataFrame(token_rows)
print(
    f"Verified all {len(token_df)} (arm, outcome) prompts via KevEncoder.prepare (strict=True, 0 truncations).\n"
    f"Branch token range: {token_df['branch_tokens'].min()} .. {token_df['branch_tokens'].max()} tokens"
)
print(token_df.groupby("stem")["branch_tokens"].agg(["min", "max"]).loc[["bev_choice", "snack_choice", "bev_healthyR", "snack_healthyG", "attn", "affect_worried", "control"]].to_string())

Verified all 160 (arm, outcome) prompts via KevEncoder.prepare (strict=True, 0 truncations).
Branch token range: 40 .. 1227 tokens
                min   max
stem                     
bev_choice      714  1031
snack_choice    772  1227
bev_healthyR     71    85
snack_healthyG   76    89
attn             50    84
affect_worried   51    85
control          60    94
